In [77]:
!pip install requests beautifulsoup4 yfinance

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [78]:
import pandas as pd

Analysing the given data and prepping for SynBank wallet portion estimation

In [79]:
df_trans = pd.read_csv('./csvs/transactional_banking.csv')
df_trade = pd.read_csv('./csvs/trade_finance.csv')
df_cross = pd.read_csv('./csvs/cross_border_payments.csv')

In [80]:
df_trans.head()

,transaction_id,entity_id,entity_name,sector,date,leg_type,direction,amount_zar,currency,channel,beneficiary_name,reference,memo
0,TXN40610803,E01,BHP Group,mining,2023-07-01,collections,inbound,63878.473869,ZAR,EFT,Continental Metals Trading House,INV-662227,NaN
1,TXN12547643,E11,Pepkor Holdings,consumer,2023-07-01,supplier_payments,outbound,53220.970000,ZAR,EFT,Sunrise Cold Chain Logistics,INV-591371,NaN
2,TXN37710224,E11,Pepkor Holdings,consumer,2023-07-01,supplier_payments,outbound,12309.970000,ZAR,SWIFT,Sunrise Cold Chain Logistics,PO-687736,NaN
3,TXN65618575,E11,Pepkor Holdings,consumer,2023-07-01,supplier_payments,outbound,6308.300000,ZAR,Internal Transfer,Sunrise Cold Chain Logistics,INV-139304,NaN
4,TXN50222056,E11,Pepkor Holdings,consumer,2023-07-01,supplier_payments,outbound,55346.750000,ZAR,Internal Transfer,Cape Wholesale Distributors,INV-556985,NaN


In [81]:
df_trade.head()

,instrument_id,entity_id,entity_name,sector,date,instrument_type,direction,tenor_days,value_zar,counterparty_country,commodity_or_contract_type,status,beneficiary_name,reference,memo
0,TF67401938,E01,BHP Group,mining,2023-07-01,letters_of_credit,export,60,13499920.55,United Arab Emirates,agri_produce,issued,Silverline Trading Co.,LC-471415,NaN
1,TF91455580,E01,BHP Group,mining,2023-07-01,letters_of_credit,export,365,558304.97,Switzerland,iron_ore,settled,Global Commodities Marketing,LC-266865,NaN
2,TF31370953,E10,Bid Corporation,consumer,2023-07-01,letters_of_credit,export,30,4042335.97,United Kingdom,agri_produce,settled,Pacific International Trading House,LC-946978,NaN
3,TF13634137,E10,Bid Corporation,consumer,2023-07-01,letters_of_credit,import,365,171042.89,China,platinum_group_metals,active,Silverline Resources Trading,LC-553503,NaN
4,TF86438695,E10,Bid Corporation,consumer,2023-07-01,letters_of_credit,export,120,331531.78,Netherlands,electronics,settled,Silverline Resources Trading,LC-260628,NaN


In [82]:
df_cross.head()

,transaction_id,entity_id,entity_name,sector,date,direction,currency_pair,value_zar,counterparty_country,corridor_type,beneficiary_name,reference,memo
0,XBP63220455,E01,BHP Group,mining,2023-07-01,inbound,USD/ZAR,2541553.55,Switzerland,intercompany,BHP Group Switzerland Ltd,INTERCO-730855,NaN
1,XBP14207725,E11,Pepkor Holdings,consumer,2023-07-01,outbound,USD/ZAR,407839.87,Angola,intercompany,Pepkor Holdings Angola Ltd,INTERCO-827118,NaN
2,XBP66460952,E11,Pepkor Holdings,consumer,2023-07-01,outbound,CNY/ZAR,72148.47,Angola,intercompany,Pepkor Holdings Angola Ltd,INTERCO-488519,NaN
3,XBP45973312,E11,Pepkor Holdings,consumer,2023-07-01,outbound,GBP/ZAR,53285.65,Namibia,intercompany,Pepkor Holdings Namibia Ltd,INTERCO-585129,NaN
4,XBP13173829,E11,Pepkor Holdings,consumer,2023-07-01,inbound,AED/ZAR,2858193.91,Japan,trade,Continental Resources Trading,TRADE-568825,NaN


In [83]:
df_trans['entity_name'].value_counts()

entity_name
Pepkor Holdings            933425
Sanlam                     611637
Bid Corporation            223301
MTN Group                  211227
Shoprite Holdings          203157
BHP Group                  125272
The Bidvest Group          108037
Aspen Pharmacare           107072
Anglo American              94535
Prosus                      49903
Naspers                     32824
AngloGold Ashanti           23915
Gold Fields                 21848
Glencore                    20930
OUTsurance Group            13835
Clicks Group                 9060
Vodacom Group                7879
NEPI Rockcastle              2796
Shaftesbury Capital plc      1425
Valterra Platinum             797
Name: count, dtype: int64

In [84]:
df_trans['entity_name'].nunique(), df_trade['entity_name'].nunique(), df_cross['entity_name'].nunique()

(20, 20, 20)

In [85]:
(df_trade['date'].min(), df_trade['date'].max()),\
(df_trans['date'].min(), df_trans['date'].max()),\
(df_cross['date'].min(), df_cross['date'].max())

(('2023-07-01', '2026-06-30'),
 ('2023-07-01', '2026-06-30'),
 ('2023-07-01', '2026-06-30'))

In [86]:
trans_wallet = df_trans[df_trans['date'].between('2025-06-30', '2026-06-30')].groupby(['entity_id', 'entity_name'])[['amount_zar']].sum().reset_index()
trade_wallet = df_trade[df_trade['date'].between('2025-06-30', '2026-06-30')].groupby(['entity_id', 'entity_name'])[['value_zar']].sum().reset_index()
cross_wallet = df_cross[df_cross['date'].between('2025-06-30', '2026-06-30')].groupby(['entity_id', 'entity_name'])[['value_zar']].sum().reset_index()

In [87]:
df_syn_wallet = trans_wallet.copy()
df_syn_wallet.rename(columns={'amount_zar': 'syn_transactional'}, inplace=True)

df_syn_wallet['syn_trade_finance'] = trade_wallet['value_zar']
df_syn_wallet['syn_cross_border'] = cross_wallet['value_zar']


In [88]:
df_syn_wallet['syn_wallet'] = df_syn_wallet[['syn_transactional', 'syn_trade_finance', 'syn_cross_border']].sum(axis=1)

In [89]:
df_syn_wallet

,entity_id,entity_name,syn_transactional,syn_trade_finance,syn_cross_border,syn_wallet
0,E01,BHP Group,1.689855e+10,1.445825e+09,2.093881e+09,2.043825e+10
1,E02,Glencore,4.461484e+09,1.085074e+09,2.329293e+09,7.875851e+09
2,E03,Anglo American,7.614543e+09,6.358514e+08,1.001065e+09,9.251459e+09
3,E04,AngloGold Ashanti,1.992849e+09,5.176479e+08,8.553962e+08,3.365893e+09
4,E05,Gold Fields,1.741836e+09,6.970445e+08,9.096257e+08,3.348507e+09
5,E06,Valterra Platinum,5.254453e+07,1.202122e+08,2.040070e+08,3.767637e+08
6,E07,OUTsurance Group,3.467256e+08,8.067232e+07,1.141805e+09,1.569203e+09
7,E08,Sanlam,1.941846e+10,1.560921e+08,2.672882e+09,2.224744e+10
8,E09,Shoprite Holdings,1.066594e+10,1.562150e+09,3.370660e+09,1.559875e+10
9,E10,Bid Corporation,1.300691e+10,1.894941e+09,6.215421e+09,2.111727e+10


In [ ]:
df_syn_wallet.to_csv('./csvs/syn_wallets.csv', index=True)

PDFs from the 20 companies will be scraped via the below links.

In [113]:
ir_pages = {
    "BVT": "https://bidvest.com/financial-results-archive",
    "BHP": "https://www.bhp.com/investor-hub/reports-and-presentations/annual-report",
    "GLE": "https://www.glencore.com/publications",
    "AGL": "https://www.angloamerican.com/investors/annual-reporting/reports-library",
    "ANG": "https://www.anglogoldashanti.com/investors/reporting/annual-reports/",
    "GFI": "https://www.goldfields.com/financial-reports.php",
    "VAL": "https://www.valterraplatinum.com/investors/annual-reporting",
    "OUT": "https://group.outsurance.co.za/results-and-reports/",
    "SLM": "https://www.sanlam.com/financial-reporting",
    "SHP": "https://www.shopriteholdings.co.za/shareholders-investors.html",
    "BID": "https://www.bidcorp-reports.com/reports/integrated-report-2025/home.php",
    "PPH": "https://pepkor.co.za/",
    "CLS": "https://www.clicksgroup.co.za/results/",
    "NRP": "https://nepirockcastle.com/investors/",
    "PRX": "https://www.prosus.com/investors/financial-information/annual-reports",
    "NPN": "https://www.naspers.com/investors/results-reports-events",
    "MTN": "https://www.mtn.com/investors/",
    "VOD": "https://www.vodacom.com/integrated-reports.php",
    "APN": "https://www.aspenpharma.com/investor-relations/",
    "SHC": "https://www.shaftesburycapital.com/en/investors/results-reports-presentations.html",
}

In [114]:

import requests
from bs4 import BeautifulSoup
import time
from urllib.parse import urljoin
import os
import re
from pathlib import Path

In [115]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
    }

def get_pdf_links(url):
    resp = requests.get(url, headers=headers, timeout=15)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')
    
    links = []
    
    for a in soup.find_all('a', href=True):
        href = a['href']
        
        if href.lower().endswith('pdf'):
            href = urljoin(url, href)
            links.append({'url': href, 'text': a.get_text(strip=True)})
            
    return links

In [116]:
def filter_report_links(links, years=('2023', '2024', '2025'), kws = ['annual report', 'integrated report', 'annual financial statements', 'afs']):
    keep = []
    
    for link in links:
        text = link['text'].lower()
        if any(y in text for y in years):
            if any(kw in text for kw in kws):
                keep.append(link)
                
    return keep

In [117]:
def download_pdf(url, out_path):
    resp = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()
    
    content = resp.content
    
    if not content.startswith(b'%PDF-'):
        print(f'WARNING: Did not return real pdf. Size {len(content)}')
        return False
    
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, 'wb') as f:
        f.write(content)
        
    return True
    

In [118]:
def sanitise_filename(text, max_len=60):
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'[<>:"/\\|?*\[\]]', '', text)
    return text[:max_len]

In [119]:
def scrape_all(ir_pages, out_dir='./scraped_pdfs'):
    resp = []
    
    for ticker, url in ir_pages.items():
        try:
            links = get_pdf_links(url)
            reports = filter_report_links(links)
            
            for report in reports:
                fname = f"{ticker}_{sanitise_filename(report['text'][:40])}.pdf".replace('/', '-')
                out_path = os.path.join(out_dir, ticker, fname)
                ok = download_pdf(report['url'], out_path)
                
                resp.append({'ticker': ticker,
                            'url': report['url'],
                            'path': out_path,
                            'success': ok})
                
        except Exception as e:
            print(f'FAILED for {ticker}: {e}')    
            resp.append({'ticker': ticker,
                          'url': url,
                          'path': None,
                          'success': False,
                          'error': str(e)})
            
    return resp

In [120]:
links = get_pdf_links(ir_pages['BVT'])
reports = filter_report_links(links)

reports

[{'url': 'https://bidvest.com/pdf\\results\\annual-results\\2025\\bidvest-group-afs.pdf',
  'text': 'FY2025 Bidvest Group\r\n\t\t\t\t\t\t\t\t\t\t\tAFS[PDF - 6MB]'},
 {'url': 'https://bidvest.com/pdf\\results\\annual-results\\2025\\bidvest-company-afs.pdf',
  'text': 'FY2025 Bidvest Company\r\n\t\t\t\t\t\t\t\t\t\t\tAFS[PDF - 2MB]'},
 {'url': 'https://bidvest.com/pdf/results/annual-results/2024/the-bidvest-group-limited-2024-consolidated-afs-no-signatures.pdf',
  'text': 'FY2024 Bidvest Group\r\n\t\t\t\t\t\t\t\t\t\t\tAFS[PDF - 3MB]'},
 {'url': 'https://bidvest.com/pdf/results/annual-results/2024/2024-the-bidvest-group-limited-company-website.pdf',
  'text': 'FY2024 Bidvest\r\n\t\t\t\t\t\t\t\t\t\t\tCompany AFS[PDF - 3.6MB]'}]

In [121]:
scrape_all(ir_pages)

FAILED for BVT: [Errno 13] Permission denied: './scraped_pdfs\\BVT\\BVT_FY2024 Bidvest Group AFSPDF.pdf'
FAILED for BHP: 403 Client Error: Forbidden for url: https://www.bhp.com/investor-hub/reports-and-presentations/annual-report
FAILED for GFI: HTTPSConnectionPool(host='www.goldfields.com', port=443): Max retries exceeded with url: /financial-reports.php (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)')))
FAILED for VAL: 404 Client Error: Not Found for url: https://www.valterraplatinum.com/investors/annual-reporting


[{'ticker': 'BVT',
  'url': 'https://bidvest.com/pdf\\results\\annual-results\\2025\\bidvest-group-afs.pdf',
  'path': './scraped_pdfs\\BVT\\BVT_FY2025 Bidvest Group AFSPDF.pdf',
  'success': False},
 {'ticker': 'BVT',
  'url': 'https://bidvest.com/pdf\\results\\annual-results\\2025\\bidvest-company-afs.pdf',
  'path': './scraped_pdfs\\BVT\\BVT_FY2025 Bidvest Company AFSP.pdf',
  'success': False},
 {'ticker': 'BVT',
  'url': 'https://bidvest.com/financial-results-archive',
  'path': None,
  'success': False,
  'error': "[Errno 13] Permission denied: './scraped_pdfs\\\\BVT\\\\BVT_FY2024 Bidvest Group AFSPDF.pdf'"},
 {'ticker': 'BHP',
  'url': 'https://www.bhp.com/investor-hub/reports-and-presentations/annual-report',
  'path': None,
  'success': False,
  'error': '403 Client Error: Forbidden for url: https://www.bhp.com/investor-hub/reports-and-presentations/annual-report'},
 {'ticker': 'GLE',
  'url': 'https://www.glencore.com/.rest/api/v1/documents/static/9b103e11-72e7-40bf-ae7c-eabe

PDFs are analysed individually after they have been scraped, to extract necessary numbers to calculate wallet.

In [100]:
FIELDS = [
    "ticker", "entity_name", "fiscal_year_end", "reporting_currency",
    "statement_type", "source_file",
    "revenue_zar", "cost_of_revenue_zar", "gross_profit_zar",
    "inventories_zar",
    "revenue_domestic_zar", "revenue_international_zar",
    "total_borrowings_zar", "finance_charges_zar",
    "extraction_confidence"
]

In [101]:
def proxies_df(file='./csvs/financial_proxies.csv'):
    # if Path(file).is_file():
    #     return pd.read_csv(file)
    
    return pd.DataFrame(columns=FIELDS)

def add_row(df, row):
    full = {f: [row.get(f)] for f in FIELDS}
    return pd.concat([df, pd.DataFrame(full)], ignore_index=True)


def save(df, file='./csvs/financial_proxies.csv'):
    df.to_csv(file, index=True)


In [102]:
def download_pdf(url, out_path):
    resp = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()
    
    content = resp.content
    
    if not content.startswith(b'%PDF-'):
        print(f'WARNING: Did not return real pdf. Size {len(content)}')
        return False
    
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, 'wb') as f:
        f.write(content)
        
    return True
    

In [103]:
df_extracted = proxies_df()

In [104]:
bvt_fy2024 = {
    "ticker": "BVT",
    "entity_name": "The Bidvest Group",
    "fiscal_year_end": "2024-06-30",
    "reporting_currency": "ZAR",
    "statement_type": "consolidated",
    "source_file": "BVT_FY2024_Bidvest_Group_AFSPDF.pdf",
    "revenue_zar": 122_615_986_000, #est wallet
    "cost_of_revenue_zar": 87_739_492_000, #est wallet
    "gross_profit_zar": 34_876_494_000,
    "inventories_zar": 14_894_387_000,
    "revenue_domestic_zar": 97_234_304_000,
    "revenue_international_zar": 31_471_636_000, #est
    "total_borrowings_zar": 31_805_322_000, 
    "finance_charges_zar": 2_506_296_000,
    "extraction_confidence": 0.98,
}

In [105]:
df_extracted = add_row(df_extracted, bvt_fy2024)

C:\Users\hiro6\AppData\Local\Temp\ipykernel_32512\1489969525.py:9: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([df, pd.DataFrame(full)], ignore_index=True)


In [106]:
save(df_extracted)

In [107]:
df_wallet = df_extracted[['ticker', 'entity_name']].copy()

In [108]:
df_wallet['transactional'] = df_extracted['revenue_domestic_zar']
df_wallet['trade_finance'] = df_extracted['cost_of_revenue_zar']
df_wallet['cross_border'] = df_extracted['revenue_international_zar'] 
df_wallet['lending'] = df_extracted['total_borrowings_zar']

In [109]:
df_wallet['wallet'] = df_wallet[['transactional', 'trade_finance', 'cross_border', 'lending']].sum(axis=1)

In [110]:
df_wallet

,ticker,entity_name,transactional,trade_finance,cross_border,lending,wallet
0,BVT,The Bidvest Group,97234304000,87739492000,31471636000,31805322000,248250754000


In [ ]:
df_wallet.to_csv('./csvs/wallets.csv', index=True)

In [123]:
df_summary = df_syn_wallet.merge(df_wallet, how='left', on='entity_name')[['entity_id', 'entity_name', 'syn_wallet', 'wallet', 'ticker']]

df_summary['wallet_gap'] = df_summary['wallet'] - df_summary['syn_wallet']
df_summary['syn_proportion'] = df_summary['syn_wallet']/df_summary['wallet']

df_summary

,entity_id,entity_name,syn_wallet,wallet,ticker,wallet_gap,syn_proportion
0,E01,BHP Group,2.043825e+10,NaN,NaN,NaN,NaN
1,E02,Glencore,7.875851e+09,NaN,NaN,NaN,NaN
2,E03,Anglo American,9.251459e+09,NaN,NaN,NaN,NaN
3,E04,AngloGold Ashanti,3.365893e+09,NaN,NaN,NaN,NaN
4,E05,Gold Fields,3.348507e+09,NaN,NaN,NaN,NaN
5,E06,Valterra Platinum,3.767637e+08,NaN,NaN,NaN,NaN
6,E07,OUTsurance Group,1.569203e+09,NaN,NaN,NaN,NaN
7,E08,Sanlam,2.224744e+10,NaN,NaN,NaN,NaN
8,E09,Shoprite Holdings,1.559875e+10,NaN,NaN,NaN,NaN
9,E10,Bid Corporation,2.111727e+10,NaN,NaN,NaN,NaN


In [ ]:
df_summary.to_csv('./csvs/wallet_summaries.csv', index=True)